## Prediction of Customer Churn Pbobability

**Version 1.0**

The notebook in fourth entry to competition I started with Basics to start with using Random Fore7r, before I test pipelione and ensemble

**Version 2.0**

Random Forests tend to struggle with this because they average predictions from many trees, which often pushes probabilities away from 0 and 1 toward the center.
This version introduces Random Forest Calibration By applying Platt Scaling (Sigmoid calibration), turns a mathematical output into a reliable financial metric for churn prevention strategies.

**Version 3.0**

Introduced GridSearchCV: It no longer guesses the number of trees; it tests 50, 100, 150, 200, 250 and 300 to find the most accurate estimator for specific data.

**Version 4.0**

Introduced notebook pipeline using Balanced Random Forest (from imblearn) designed to:
- Handle churn imbalance properly
- Increase churn probability sensitivity
- Optimize for ROC-AUC
- Improve recall for churners
- Provide calibrated probabilities
- Use stratified CV
- Automatically tune threshold

**Version 5.0***

Added multi-seed ensemble combining XGBoost + Random Forest, optimized for churn imbalance and probability separation.
- XGBoost with scale_pos_weight
- RandomForest with class_weight='balanced'
- 3 seeds
- 5-fold stratified CV
- Rank normalization blending
- Weight optimization
- Threshold tuning for recall

**Version 6.0**

Added CatBoost to XGBoost and RF
  3-model multi-seed ensemble:
- XGBoost
- Random Forest
- CatBoost
- 3 seeds
- 5-fold stratified CV
- Rank blending
- Weight optimization
- Threshold tuning

**Version 7.0**


Added LightGBM to the previous XGBoost + Random Forest + CatBoost multi-seed ensemble to evaluate 4 model ensemble, these four specific algorithms leverage different mathematical "perspectives" on your data:
- XGBoost & LightGBM: These are "Gradient Boosted Decision Trees" (GBDT). They build trees sequentially, where each tree tries to correct the errors of the previous one. They are highly aggressive and excellent at finding complex patterns.
- CatBoost: It uses a specialized technique called "Ordered Boosting" which is specifically designed to reduce bias when dealing with categorical variables. It often finds insights that XGB/LGB miss.
- Random Forest: This is a "Bagging" model. Unlike the others, it builds many independent trees in parallel and averages them. It is much harder to overfit, making it a perfect "anchor" for your ensemble to prevent the boosting models from becoming too volatile.

**Version 8.0**

Added Feature Engineering along with automatic ensemble weight calculation, the model now has
- Feature Engineering (tenure ratios, charge ratios, service counts)
- Automatic Ensemble Weight Search
- Multi-seed ensemble (XGB + LGBM + CatBoost + RF)
- Rank blending
- Grid search for optimal ensemble weights

**Version 9.0**

Improved pipeline by adding
- Automatic feature generation
- Multi-seed base models (XGB + LGBM + CatBoost + RF)
- Out-of-fold predictions
- Stacking meta-model (Logistic Regression)
- Rank normalization

The pipeline can be depicted:

- Raw Data
- Feature Engineering
- Base Models (Level-1)

--   ├── XGBoost  
--   ├── LightGBM  
--   ├── CatBoost
--   └── RandomForest

- OOF Predictions
- Stacking Dataset
- Meta Model (Level-2)
- Logistic Regression
- Final Predictions

**Version 10.0**
 - Raw Data
 - Feature Engineering
 - Encoding + Scaling
 - Level-1 Models
   -- ── XGBoost
   -- ── LightGBM
   -- ── CatBoost
 - OOF Predictions
 - Pseudo-Labeling
 - Level-2 Stacking
   -- ── Ridge
   -- ── Logistic
   -- ── Neural Network
 - Final Ensemble

**Version 11.0**
 - Raw Data
 - Feature Engineering
 - Encoding + Scaling
 - Level-1 Multi Seed Models 
   -- ── XGBoost
   -- ── LightGBM
   -- ── CatBoost
 - OOF Predictions
 - Pseudo-Labeling
 - Level-2 Stacking
   -- ── Ridge
   -- ── Logistic
   -- ── Neural Network
 - Final Ensemble

In [1]:
# ============================================================
# 1. Imports
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier
import warnings

# Suppress the specific Pandas FutureWarning
warnings.filterwarnings("ignore", category=FutureWarning, message=".*Setting an item of incompatible dtype.*")

# Suppress the Scikit-learn UserWarning about feature names
warnings.filterwarnings("ignore", category=UserWarning, message=".*X does not have valid feature names.*")

In [2]:
# =============================================================================
# 2. Data Loading 
# =============================================================================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

TARGET = "Churn"
ID_COL = "id"
train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}) #
y = train[TARGET]
test_ids = test[ID_COL]

X = train.drop([TARGET, ID_COL], axis=1)
X_test = test.drop([ID_COL], axis=1)

# Categorical Encoding
cat_cols = X.select_dtypes(include=["object"]).columns
for col in cat_cols:
    le = LabelEncoder()
    full_data = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full_data)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

In [3]:
# ============================================================
# 3. Automatic Feature Engineering
# ============================================================

def feature_engineering(df):
    df["avg_monthly_charge"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["charge_ratio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)
    return df

X = feature_engineering(X)
X_test = feature_engineering(X_test)

# Polynomials on numeric features only
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "avg_monthly_charge"]
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X[num_cols])
X_poly_test = poly.transform(X_test[num_cols])

X = np.hstack([X.values, X_poly])
X_test = np.hstack([X_test.values, X_poly_test])

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

In [4]:
# ============================================================
# 5. Initialize OOF container + Level 1 Training
# ============================================================
N_FOLDS = 9
SEEDS = [11, 42, 99] 
models_list = ["xgb", "lgb", "cat"]

oof_df = pd.DataFrame(0.0, index=range(len(X)), columns=models_list)
pred_df = pd.DataFrame(0.0, index=range(len(X_test)), columns=models_list)

total_iterations = N_FOLDS * len(SEEDS)

for seed in SEEDS:
    print(f"\n{'#'*20} RUNNING SEED {seed} {'#'*20}")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        # --- XGBoost ---
        m_xgb = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.025, max_depth=6, random_state=seed, tree_method='hist')
        m_xgb.fit(X_tr, y_tr)
        p_val_xgb = m_xgb.predict_proba(X_val)[:, 1]
        oof_df.loc[val_idx, "xgb"] += p_val_xgb / len(SEEDS)
        pred_df["xgb"] += m_xgb.predict_proba(X_test)[:, 1] / total_iterations
        print(f"Seed {seed} | Fold {fold+1} | XGB AUC: {roc_auc_score(y_val, p_val_xgb):.5f}")

        # --- LightGBM ---
        m_lgb = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.025, verbose=-1, random_state=seed)
        m_lgb.fit(X_tr, y_tr)
        p_val_lgb = m_lgb.predict_proba(X_val)[:, 1]
        oof_df.loc[val_idx, "lgb"] += p_val_lgb / len(SEEDS)
        pred_df["lgb"] += m_lgb.predict_proba(X_test)[:, 1] / total_iterations
        print(f"Seed {seed} | Fold {fold+1} | LGB AUC: {roc_auc_score(y_val, p_val_lgb):.5f}")

        # --- CatBoost ---
        m_cat = CatBoostClassifier(iterations=5000, learning_rate=0.025, verbose=False, random_seed=seed)
        m_cat.fit(X_tr, y_tr)
        p_val_cat = m_cat.predict_proba(X_val)[:, 1]
        oof_df.loc[val_idx, "cat"] += p_val_cat / len(SEEDS)
        pred_df["cat"] += m_cat.predict_proba(X_test)[:, 1] / total_iterations
        print(f"Seed {seed} | Fold {fold+1} | CAT AUC: {roc_auc_score(y_val, p_val_cat):.5f}")
    


#################### RUNNING SEED 11 ####################
Seed 11 | Fold 1 | XGB AUC: 0.91542
Seed 11 | Fold 1 | LGB AUC: 0.91574
Seed 11 | Fold 1 | CAT AUC: 0.91637
Seed 11 | Fold 2 | XGB AUC: 0.91408
Seed 11 | Fold 2 | LGB AUC: 0.91469
Seed 11 | Fold 2 | CAT AUC: 0.91572
Seed 11 | Fold 3 | XGB AUC: 0.91413
Seed 11 | Fold 3 | LGB AUC: 0.91419
Seed 11 | Fold 3 | CAT AUC: 0.91514
Seed 11 | Fold 4 | XGB AUC: 0.91708
Seed 11 | Fold 4 | LGB AUC: 0.91732
Seed 11 | Fold 4 | CAT AUC: 0.91795
Seed 11 | Fold 5 | XGB AUC: 0.91582
Seed 11 | Fold 5 | LGB AUC: 0.91611
Seed 11 | Fold 5 | CAT AUC: 0.91692
Seed 11 | Fold 6 | XGB AUC: 0.91683
Seed 11 | Fold 6 | LGB AUC: 0.91700
Seed 11 | Fold 6 | CAT AUC: 0.91787
Seed 11 | Fold 7 | XGB AUC: 0.91526
Seed 11 | Fold 7 | LGB AUC: 0.91626
Seed 11 | Fold 7 | CAT AUC: 0.91706
Seed 11 | Fold 8 | XGB AUC: 0.91581
Seed 11 | Fold 8 | LGB AUC: 0.91601
Seed 11 | Fold 8 | CAT AUC: 0.91708
Seed 11 | Fold 9 | XGB AUC: 0.91530
Seed 11 | Fold 9 | LGB AUC: 0.91553
Seed 

In [5]:
# ============================================================
# 3. Rank Normalization Step
# ============================================================
print("\nApplying Rank Normalization to Level-1 Predictions...")
oof_rank = oof_df.rank(pct=True)
pred_rank = pred_df.rank(pct=True)


Applying Rank Normalization to Level-1 Predictions...


In [6]:
# ============================================================
# 7. Level 2 - Model
# ============================================================
meta_models = {
    "ridge": Ridge(alpha=1.0),
    "log": LogisticRegression(),
    "nn": MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
}

meta_oof = pd.DataFrame(0.0, index=range(len(X)), columns=meta_models.keys())
meta_pred = pd.DataFrame(0.0, index=range(len(X_test)), columns=meta_models.keys())

# Meta-level K-Fold
skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

for name, model in meta_models.items():
    print(f"Training Meta-Model: {name}")
    for fold, (tr, val) in enumerate(skf_meta.split(oof_rank, y)):
        X_tr_m, X_val_m = oof_rank.iloc[tr], oof_rank.iloc[val]
        y_tr_m = y.iloc[tr]
        
        model.fit(X_tr_m, y_tr_m)
        
        if name == "ridge":
            meta_oof.loc[val, name] = model.predict(X_val_m)
            meta_pred[name] += model.predict(pred_rank) / N_FOLDS
        else:
            meta_oof.loc[val, name] = model.predict_proba(X_val_m)[:, 1]
            meta_pred[name] += model.predict_proba(pred_rank)[:, 1] / N_FOLDS

Training Meta-Model: ridge
Training Meta-Model: log
Training Meta-Model: nn


In [7]:
# ============================================================
# 8. Final Ensemble
# ============================================================

final_oof = (0.4 * meta_oof["log"] + 0.3 * meta_oof["ridge"] + 0.3 * meta_oof["nn"])
final_test = (0.4 * meta_pred["log"] + 0.3 * meta_pred["ridge"] + 0.3 * meta_pred["nn"])

print(f"\nFinal Stacked (Ranked) AUC: {roc_auc_score(y, final_oof):.5f}")



Final Stacked (Ranked) AUC: 0.91695


In [8]:
# =============================================================================
# 9. Submission
# =============================================================================
submission = pd.DataFrame({
    "id":test_ids,
    "Churn":final_test
})

submission.to_csv("submission.csv", index=False)

submission.head(20)




,id,Churn
0,594194,0.129096
1,594195,-0.056881
2,594196,0.160788
3,594197,-0.021828
4,594198,0.519394
5,594199,0.243545
6,594200,0.806815
7,594201,-0.021917
8,594202,0.075516
9,594203,0.355250
